# PPI-KG
+ GE & PPI-KG overlapping proteins: 8085
+ GE proteins: 8085 + 12480 = 20565
+ PPI-KG proteins: 8085 + 543 = 8628

### BUT the PPI-KG I have can't match the same protein counts  

# Sample Scoring
+ ecdf;
+ healthy control as reference distribution;
+ 1%, 1.5%, 2.5%, 5%, 10%, 20%

# Generate Network
+ my PNG.generate() generate the same results as CLEP's network generator

In [1]:
import os
import sys

try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))

from data_processing.pyg_graph_generator import *

/home/xyu/.conda/envs/firegnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
output_dir = "../CLEP_repeat/networks/PPI_KGs_my"
dataset = 'adni'
process_method = 'DiseaseKG'
scoring_method = 'ecdf1'

kg_disease_path = "../datasets/base_kgs/ppi_hc.pkl"
kg_health_path = "../data/KG/healthy_aging_reversed_remove_noncausal.pkl"
exp_path = "../data/ADNI/cleaned_gene_expression_data.csv"
scoring_path = "../data/ADNI/old_target/ecdf_1/sample_scoring_ecdf.csv"

os.makedirs(output_dir, exist_ok=True)
save_network = os.path.join(output_dir, f"G_{dataset}_{process_method}_{scoring_method}.pkl")
save_summary = os.path.join(output_dir, f"Summary_{dataset}_{process_method}_{scoring_method}.csv")

In [4]:
# 1. Load expression df, smaple scoring df, KG
exp_df = pd.read_csv(exp_path, index_col=0)
if exp_df.shape[0] > exp_df.shape[1]:
    exp_df = exp_df.transpose()
data = pd.read_csv(scoring_path, index_col=0)
kg_disease = load_graph(kg_disease_path)
kg_control = load_graph(kg_health_path)

# clean exp_df before K-NN
# drop genes with no variation
exp_df = exp_df.loc[:, exp_df.std() > 0]
# Using median is usually safer for gene expression
exp_df = exp_df.fillna(exp_df.median())
# normalize safely
min_val = exp_df.min()
max_val = exp_df.max()
exp_norm = (exp_df - min_val) / (max_val - min_val + 1e-9)
# final fill-na
exp_norm = exp_norm.fillna(0)

Loaded graph from ../datasets/base_kgs/ppi_hc.pkl: 17042 nodes, 382526 edges
Loaded graph from ../data/KG/healthy_aging_reversed_remove_noncausal.pkl: 4161 nodes, 13775 edges


# RotatE 
+ best hyperparameters are in the config dict

In [48]:
2**6

64

In [ ]:
# best hyperparameters for RotatE

best_config = {
    "model": "RotatE",
    "training_loop": "sLCWA",  # Stochastic Local Closed World Assumption
    
    # Model-specific parameters
    "model_kwargs": {
        "embedding_dim": 128,  
    },
    
    # Loss function and its specific parameters
    "loss": "NSSALoss",  # Negative Sampling Self-Adversarial Loss
    "loss_kwargs": {
        "margin": 23.92,
        "adversarial_temperature": 0.93,
    },
    
    # Negative Sampler configuration (used by sLCWA)
    "negative_sampler_kwargs": {
        "num_negatives_per_positive": 22,
    },
    
    # Training runtime execution parameters
    "training_kwargs": {
        "num_epochs": 200,
        "batch_size": 1024,
    }
}

In [7]:
from CLEP_repeat.embedding.kge import *

In [8]:
graph = load_graph("../CLEP_repeat/networks/PPI_KGs/G_adni_OldTarget_DiseaseKG_ecdf_1.pkl")
design_path = "../data/ADNI/design_with_real_target.tsv"
design = pd.read_csv(design_path, index_col=0, sep='\t')
design['Target'] = design['Old_Target'].map({'Disease':1, 'Control':0})
design

Loaded graph from ../CLEP_repeat/networks/PPI_KGs/G_adni_OldTarget_DiseaseKG_ecdf_1.pkl: 17786 nodes, 994402 edges


,Old_Target,Target
FileName,,
116_S_1249,Control,0
037_S_4410,Control,0
006_S_4153,Disease,1
116_S_1232,Control,0
099_S_4205,Disease,1
...,...,...
009_S_2381,Disease,1
053_S_4557,Disease,1
073_S_4300,Disease,1


In [9]:
graph_df = nx.to_pandas_edgelist(graph)
graph_df

,source,target,confidence,weight,label,evidence,relation
0,AL1A1_HUMAN,AL1A1_HUMAN,0.76,NaN,NaN,"experiments:in vivo,Two-hybrid;pmids:12081471,...",PPI
1,AL1A1_HUMAN,AL1A1_HUMAN,0.76,NaN,NaN,"experiments:in vivo,Two-hybrid;pmids:12081471,...",rev_PPI
2,AL1A1_HUMAN,ALDH2_HUMAN,0.82,NaN,NaN,"experiments:Two-hybrid,cross-linking study;pmi...",PPI
3,AL1A1_HUMAN,IL21_HUMAN,0.72,NaN,NaN,experiments:affinity chromatography technology...,rev_PPI
4,AL1A1_HUMAN,ITLN2_HUMAN,0.72,NaN,NaN,experiments:affinity chromatography technology...,rev_PPI
...,...,...,...,...,...,...,...
994397,007_S_0101,YPEL2_HUMAN,NaN,0.127827,1.0,NaN,down_reg
994398,007_S_0101,ZBP1_HUMAN,NaN,0.862738,1.0,NaN,up_reg
994399,007_S_0101,ZFP41_HUMAN,NaN,0.739417,1.0,NaN,up_reg
994400,007_S_0101,ZNFX1_HUMAN,NaN,0.895191,1.0,NaN,up_reg


In [10]:
graph_df=graph_df[['source','target','relation','label']]
graph_df = graph_df[~graph_df['relation'].str.contains('rev', na=False)]
graph_df

,source,target,relation,label
0,AL1A1_HUMAN,AL1A1_HUMAN,PPI,NaN
2,AL1A1_HUMAN,ALDH2_HUMAN,PPI,NaN
16,AL1A1_HUMAN,AL1A2_HUMAN,PPI,NaN
30,AL1A1_HUMAN,P4K2A_HUMAN,PPI,NaN
32,ITA7_HUMAN,ACHA_HUMAN,PPI,NaN
...,...,...,...,...
994397,007_S_0101,YPEL2_HUMAN,down_reg,1.0
994398,007_S_0101,ZBP1_HUMAN,up_reg,1.0
994399,007_S_0101,ZFP41_HUMAN,up_reg,1.0
994400,007_S_0101,ZNFX1_HUMAN,up_reg,1.0


### KGE

In [11]:
def weighted_splitter(
        edgelist: pd.DataFrame,
        train_size: float = 0.8,
        validation_size: float = 0.1
) -> Tuple[pd.DataFrame, ...]:
    """Split the given edgelist into training, validation and testing sets on the basis of the ratio of relations.

    :param edgelist: Edgelist in the form of (Source, Relation, Target)
    :param train_size: Size of the training data
    :param validation_size: Size of the training data
    :return: Tuple containing the train, validation & test splits
    """
    # Validation size is the size of the percentage of the remaining data (i.e. If required validation size is 10% of
    # the original data & training size is 80% then the new validation size is 50% of the data without the training
    # data. The similar calculation is done for training size, hence it is always 1
    validation_size = validation_size / (1 - train_size)
    test_size = 1

    # Get the unique relations in the network
    unique_relations = sorted(edgelist['relation'].unique())

    data = edgelist.drop_duplicates().copy()

    split = []
    # Split the data to get training, validation and test samples
    for frac_size in [train_size, validation_size, test_size]:
        frames = []
        # Random sampling of the data for every type of relation
        for relation in unique_relations:
            temp = data[data['relation'] == relation].sample(frac=frac_size)

            data = data[~data.index.isin(temp.index)]

            frames.append(temp)
        # Join all the different relations in one dataframe
        split.append(pd.concat(frames, ignore_index=True, sort=False))

    return tuple(split)

In [12]:
hpo_config = {
    # 1. Wrap all model and training specs into the "pipeline" key
    "pipeline": {
        "model": "RotatE",
        "training_loop": "sLCWA",
        
        "model_kwargs": {
            "embedding_dim": 128,  
        },
        
        "loss": "NSSALoss",
        "loss_kwargs_ranges": {
            "margin": {
                "type": "float",
                "low": 15.0,
                "high": 30.0,
                "scale": "linear"
            },
            "adversarial_temperature": {
                "type": "float",
                "low": 0.5,
                "high": 1.5,
                "scale": "linear"
            }
        },

        "negative_sampler": "basic",
        "negative_sampler_kwargs_ranges": {
            "num_negs_per_pos": {
                "type": "int",
                "low": 10,
                "high": 30,
                "step": 2
            }
        },

        "optimizer": "Adam",
        "optimizer_kwargs_ranges": {
            "lr": {
                "type": "float",
                "low": 0.0001,
                "high": 0.01,
                "scale": "log"
            }
        },

        "training_kwargs": {
            "num_epochs": 20,
        },
        "training_kwargs_ranges": {
            "batch_size": {
                "type": "categorical",
                "choices": [512, 1024, 2048]
            }
        },

        "evaluator": "RankBasedEvaluator",
        "evaluator_kwargs": {
            "filtered": True
        },
        "evaluation_kwargs": {
            "batch_size": 1024
        }
    },
    
    # 2. mandatory "optuna" key to control the HPO mechanics
    "optuna": {
        "n_trials": 2,       # Number of parameter combinations to try
        "timeout": 3600,      # Optional: Max time in seconds (1 hour)
        "metric": "hits_at_10", # What metric to maximize (defaults to mean_reciprocal_rank if left out)
        "direction": "maximize" 
    }
}

In [13]:
edgelist = graph_df
out = './kge_result'
train_size = 0.8
validation_size = 0.1

os.makedirs(out, exist_ok=True)

In [14]:
design_norm_df = design.astype(str, copy=True)

unique_nodes = edgelist[~edgelist['label'].isna()].drop_duplicates('source')

# Create a mapping of the patient to the label. The patient id is converted to string to avoid duplicates
label_mapping = {str(patient): label for patient, label in zip(unique_nodes['source'], unique_nodes['label'])}

edgelist = edgelist.drop(columns='label')

# Split the edgelist into training, validation and testing data
train, validation, test = weighted_splitter(
    edgelist=edgelist,
    train_size=train_size,
    validation_size=validation_size
)

train_path = os.path.join(out, 'train.edgelist')
validation_path = os.path.join(out, 'validation.edgelist')
test_path = os.path.join(out, 'test.edgelist')

train.to_csv(train_path, sep='\t', index=False, header=False)
validation.to_csv(validation_path, sep='\t', index=False, header=False)
test.to_csv(test_path, sep='\t', index=False, header=False)

In [15]:
from pykeen.hpo.hpo import hpo_pipeline_from_config

train_tf = TriplesFactory.from_path(train_path, create_inverse_triples=True)
validation_tf = TriplesFactory.from_path(validation_path, create_inverse_triples=True)
test_tf = TriplesFactory.from_path(test_path, create_inverse_triples=True)

In [ ]:


# Define HPO pipeline
hpo_results = hpo_pipeline_from_config(
    dataset=None,
    training=train_tf,
    testing=test_tf,
    validation=validation_tf,
    config=hpo_config
)

optimization_dir = os.path.join(out, 'pykeen_results_optim')
if not os.path.isdir(optimization_dir):
    os.makedirs(optimization_dir)

hpo_results.save_to_directory(optimization_dir)

[I 2026-06-01 11:15:51,326] A new study created in memory with name: no-name-aaba25b3-337b-4160-9dfa-dd5feadee4ee
INFO:pykeen.hpo.hpo:Using model: <class 'pykeen.models.unimodal.rotate.RotatE'>
INFO:pykeen.hpo.hpo:Using loss: <class 'pykeen.losses.NSSALoss'>
INFO:pykeen.hpo.hpo:Using optimizer: <class 'torch.optim.adam.Adam'>
INFO:pykeen.hpo.hpo:Using training loop: <class 'pykeen.training.slcwa.SLCWATrainingLoop'>
INFO:pykeen.hpo.hpo:Using negative sampler: <class 'pykeen.sampling.basic_negative_sampler.BasicNegativeSampler'>
INFO:pykeen.hpo.hpo:Using evaluator: <class 'pykeen.evaluation.rank_based_evaluator.RankBasedEvaluator'>
INFO:pykeen.hpo.hpo:Attempting to maximize both.realistic.hits_at_10
INFO:pykeen.hpo.hpo:Filter validation triples when testing: True
INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.triples.triples_factory:Creating inv

In [16]:
import json

from pykeen.pipeline import pipeline_from_config, PipelineResult

# Retrain with best hp
config_path = os.path.join(out, 'pykeen_results_optim', 'best_pipeline', 'pipeline_config.json')
with open(config_path, 'r') as f:
    best_config_dict = json.load(f)


# Remove ALL structural keys causing duplicates or path errors
if "pipeline" in best_config_dict:
    inner_pipeline = best_config_dict["pipeline"]
    inner_pipeline.pop("dataset", None)
    inner_pipeline.pop("training", None)
    inner_pipeline.pop("testing", None)
    inner_pipeline.pop("validation", None)

# Execute training safely using your in-memory objects
pipeline_results = pipeline_from_config(
    config=best_config_dict,
    training=train_tf,
    testing=test_tf,
    validation=validation_tf
)

print("Retraining completed successfully!")
best_pipeline_dir = os.path.join(out, 'pykeen_results_final')
if not os.path.isdir(best_pipeline_dir):
    os.makedirs(best_pipeline_dir)

pipeline_results.save_to_directory(best_pipeline_dir, save_replicates=True)

No random seed is specified. Setting to 1818411026.
INFO:pykeen.triples.triples_factory:Creating inverse triples.
Training epochs on cuda:0: 100%|██████████| 2/2 [00:13<00:00,  6.61s/epoch, loss=1.24, prev_loss=5.76]
Evaluating on cuda:0: 100%|██████████| 47.7k/47.7k [00:17<00:00, 2.65ktriple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 20.39s seconds


Retraining completed successfully!


INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=15111, num_relations=31040, create_inverse_triples=True, num_triples=381282, path="/home/xyu/thesis/HybridKG/CLEP_repeat/kge_result/train.edgelist") to file:///home/xyu/thesis/HybridKG/CLEP_repeat/kge_result/pykeen_results_final/training_triples
INFO:pykeen.pipeline.api:Saved to directory: /home/xyu/thesis/HybridKG/CLEP_repeat/kge_result/pykeen_results_final


In [17]:
def model_to_numpy(
        model: ERModel[HeadRepresentation, RelationRepresentation, TailRepresentation],
        complex: bool = False
) -> npt.NDArray[np.float64 | np.float32]:
    """Retrieve embedding from the models as a numpy array."""
    embedding_numpy: npt.NDArray[np.float64 | np.float32] = model.entity_representations[0](indices=None).detach().cpu().numpy()

    if complex:
        return embedding_numpy

    # Get the real part of the embedding for classification tasks
    return embedding_numpy.real

In [ ]:
best_model, triple_factory = pipeline_results.model, pipeline_results.training

# Get the embedding as a numpy array. Ignore the type as the model will be of type ERModel (Embedding model)
embedding_values = model_to_numpy(best_model)
embedding_values.shape

# Create columns as component names
embedding_columns = [f'Component_{i}' for i in range(1, embedding_values.shape[1] + 1)]

# Get the nodes of the training triples as index
node_list = list(triple_factory.entity_to_id.keys())
embedding_index = sorted(node_list, key=lambda x: triple_factory.entity_to_id[x])

embedding = pd.DataFrame(data=embedding_values, columns=embedding_columns, index=embedding_index)
embedding
# get patient embedding
embedding = embedding[embedding.index.isin(design_norm_df.index)]

for index in embedding.index:
    embedding.at[index, 'label'] = label_mapping[index]

embedding

,Component_1,Component_2,Component_3,Component_4,Component_5,Component_6,Component_7,Component_8,Component_9,Component_10,...,Component_120,Component_121,Component_122,Component_123,Component_124,Component_125,Component_126,Component_127,Component_128,label
002_S_0413,-1.185832,-1.246140,-1.213335,1.131570,1.067649,1.202655,-1.243473,1.140229,-1.175492,-1.078351,...,0.779981,-0.133348,-1.217663,1.270100,-0.855835,1.083572,1.149201,-1.172288,1.166304,0.0
002_S_0685,-1.012053,-1.128103,1.071814,-1.179738,1.014606,-1.138972,-1.035773,-1.053765,-1.110086,1.051507,...,-1.257817,1.011820,-1.027811,-1.106328,1.105659,-1.110709,-1.024613,0.981606,1.008754,0.0
002_S_0729,-1.053835,1.153098,-1.272817,1.135944,1.149762,1.154957,1.175766,-1.159868,-0.435735,1.280673,...,-1.075501,-1.169267,-1.111074,1.194368,1.170830,-1.292662,1.071426,1.169955,-1.092085,1.0
002_S_1155,-1.146257,-1.093666,-0.990296,-1.027638,-1.081640,1.000926,-1.171302,-1.080923,1.150759,1.010814,...,1.197751,-1.146308,1.069681,1.120988,-1.109052,-1.101402,-1.048250,-1.110054,-1.090493,1.0
002_S_1261,-1.150729,-1.050795,1.108313,1.029440,1.108306,-1.071426,-0.937996,-1.018963,-1.010212,-1.054455,...,1.088933,1.056646,1.029420,-1.098254,1.084621,-1.028382,-1.095245,1.044967,1.052722,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941_S_4292,1.128455,-1.124778,-1.086440,-1.024085,0.965640,-0.968342,0.991333,-0.971577,-1.148673,-1.038322,...,-0.940778,1.154144,-1.017928,-0.965501,1.110026,1.012227,-1.097784,-1.096938,1.126387,0.0
941_S_4365,1.054377,1.074285,-1.129239,1.056024,1.048725,1.067354,-0.991954,-0.942700,-1.052738,-1.126211,...,1.076357,1.034866,-1.133501,1.102200,1.159967,1.121948,-0.982380,-1.182646,-1.044021,0.0
941_S_4376,-0.945320,1.054398,-1.184591,-0.583063,-1.035601,-1.034739,-1.058644,1.175970,-1.143681,-1.211664,...,1.164353,1.138245,-1.206443,1.152960,-1.002042,0.953729,1.151957,-1.154629,-1.131732,0.0
941_S_4377,-1.082365,-1.136318,0.965985,-1.080356,1.107801,-1.123411,1.265278,-1.128067,-1.178771,-1.074376,...,1.182036,0.938314,1.225379,1.051974,-1.120849,1.151734,-0.986342,-1.003260,1.097025,1.0


In [35]:
embedding.to_csv(os.path.join(best_pipeline_dir, 'sample_embeddings.csv'))

# Classification: Logistic Regression

In [3]:
from CLEP_repeat.classification.hpo import *

db_url = "sqlite:///optuna_study.db"

embedding = pd.read_csv("../CLEP_repeat/kge_result/pykeen_results_final/sample_embeddings.csv", index_col=0)
cv_results = do_classification(data=embedding,
                                model_name='logistic_regression',
                                out_dir = "../CLEP_repeat/kge_result/cls_result",
                                validation_cv=5,
                                scoring_metrics=['roc_auc','f1','accuracy','average_precision'],
                                rand_labels=False,
                                mysql_url=db_url,
                                num_processes=1,
                                num_trials=10)

Outer CV: 100%|██████████| 5/5 [00:02<00:00,  2.05it/s]
